In [0]:
%sql

SELECT COUNT(*) FROM bootcamp.silver.propiedades;

In [0]:
%sql

DESCRIBE TABLE bootcamp.silver.propiedades;

#Modelado Gold - Star Schema

## Modulo 1-E1.1

### dim_zona_v1

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_zona;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona (
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1)  COMMENT "PK-Identificador único de la zona",
    partido STRING NOT NULL COMMENT "Partido donde se encuentra la propiedad",
    region STRING NOT NULL COMMENT "Region donde se encuentra la propiedad",
    ciudad STRING NOT NULL COMMENT "Ciudad donde se encuentra la propiedad",
    provincia STRING NOT NULL COMMENT "Provincia donde se encuentra la propiedad" DEFAULT "Buenos Aires",
    pais string NOT NULL COMMENT "Pais donde se encuentra la propiedad" DEFAULT "Argentina",
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Zona";



### dim_zona_v2 declaracion PK explicita

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.gold.dim_zona;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona (
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1)  COMMENT "PK-Identificador único de la zona",
    partido STRING NOT NULL COMMENT "Partido donde se encuentra la propiedad",
    region STRING NOT NULL COMMENT "Region donde se encuentra la propiedad",
    ciudad STRING NOT NULL COMMENT "Ciudad donde se encuentra la propiedad",
    provincia STRING NOT NULL COMMENT "Provincia donde se encuentra la propiedad" DEFAULT "Buenos Aires",
    pais string NOT NULL COMMENT "Pais donde se encuentra la propiedad" DEFAULT "Argentina",
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY(zona_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Zona";

In [0]:
%sql

DESCRIBE bootcamp.gold.dim_zona;

### dim_tipo_operacion declaracion PK explicita

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_tipo_operacion;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tipo_operacion(
    tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1)  COMMENT "PK-Identificador único de la operacion",
    tipo_operacion STRING NOT NULL COMMENT "Tipo de operacion",
    moneda STRING NOT NULL COMMENT "Tipo de moneda",
    categoria STRING NOT NULL COMMENT "Categoria",
    descripcion STRING NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY(tipo_operacion_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Tipo Operacion";

### dim_tiempo tabla estatica con declaracion PK explicita

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_tiempo;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tiempo(
    fecha_id BIGINT COMMENT "PK-Identificador único de la fecha",
    fecha DATE NOT NULL,
    anio INT NOT NULL,
    mes INT NOT NULL,
    trimestre INT NOT NULL,
    dia_semana STRING NOT NULL,
    es_fin_de_semana BOOLEAN NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() COMMENT 'DateTime in UTC',
    PRIMARY KEY(fecha_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Estatica Tiempo";

### dim_caracterisiticas con declaracion PK explicita

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_caracteristicas;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_caracteristicas(
    caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT "PK-Identificador único de las caracteristicas",
    estado STRING NOT NULL,
    cochera BOOLEAN NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() COMMENT 'DateTime in UTC',
    PRIMARY KEY(caracteristicas_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Caracteristicas";


### dim_orientacion con declaracion PK explicita

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.dim_orientacion;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_orientacion(
    orientacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT "PK-Identificador único de la orientacion",
    orientacion STRING NOT NULL,
    tipo_orientacion STRING NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() COMMENT 'DateTime in UTC',
    PRIMARY KEY(orientacion_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Orientacion";

### fact_propiedades, tabla de hechos con declaracion PK y FK explicita

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.fact_propiedades;

CREATE TABLE IF NOT EXISTS bootcamp.gold.fact_propiedades(
    row_hash STRING NOT NULL COMMENT "PK-ROW HASH único de la propiedad",
    zona_id BIGINT NOT NULL COMMENT "FK - dim_zona",
    tipo_orientacion_id BIGINT NOT NULL COMMENT "FK-dim_orientacion",
    fecha_id BIGINT NOT NULL COMMENT "FK-dim_tiempo",
    caracteristicas_id BIGINT NOT NULL COMMENT "FK-dim_caracteristicas",
    tipo_operacion_id BIGINT NOT NULL COMMENT "FK-dim_tipo_operacion",
    url STRING NOT NULL COMMENT "URL de la propiedad",
    precio DECIMAL(15,2),
    expensas DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    ambientes INT,    
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() COMMENT 'DateTime in UTC',
    PRIMARY KEY(row_hash),   
    FOREIGN KEY (zona_id)
    REFERENCES bootcamp.gold.dim_zona (zona_id),
    FOREIGN KEY (tipo_orientacion_id)
    REFERENCES bootcamp.gold.dim_orientacion (orientacion_id),
    FOREIGN KEY (fecha_id)
    REFERENCES bootcamp.gold.dim_tiempo (fecha_id),
    FOREIGN KEY (caracteristicas_id)
    REFERENCES bootcamp.gold.dim_caracteristicas (caracteristicas_id),
    FOREIGN KEY (tipo_operacion_id)
    REFERENCES bootcamp.gold.dim_tipo_operacion (tipo_operacion_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Fact Propiedades";

### validacion de que a pesar de estar declaradas las FK databricks no valida si un valor que se inserta existe en la tabla referenciada

In [0]:
%sql
INSERT INTO bootcamp.gold.fact_propiedades (
    row_hash,
    zona_id,
    tipo_orientacion_id,
    fecha_id,
    caracteristicas_id,
    tipo_operacion_id,
    url
)
VALUES (
    'abc',
    9999,
    1,
    20260101,
    1,
    1,
    'https://test.com'
);

## Migracion de SILVER a tablas GOLD

In [0]:
%sql
INSERT INTO bootcamp.gold.dim_zona (    
    partido,
    region,
    ciudad,
    provincia,
    pais
)
SELECT 
        DISTINCT 
        partido, 
        region,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'GBA'
        END as ciudad,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'Buenos Aires'
        END as provincia,
        'Argentina' as pais
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL AND region IS NOT NULL
ORDER BY partido, region;
    


In [0]:
%sql

SELECT DISTINCT tipo_operacion FROM bootcamp.silver.propiedades;

In [0]:
%sql
SELECT DISTINCT moneda FROM bootcamp.silver.propiedades;

In [0]:
%sql

INSERT INTO bootcamp.gold.dim_tipo_operacion
(
    tipo_operacion,
    moneda,
    categoria,
    descripcion
)
SELECT
    DISTINCT
    p.tipo_operacion,
    p.moneda,
    CASE
        WHEN p.tipo_operacion = 'alquiler' THEN 'Alquiler'
        WHEN p.tipo_operacion = 'venta' THEN 'Venta'
        ELSE 'Otros'
    END AS categoria,
    CASE
        WHEN p.tipo_operacion ='alquiler' THEN 'Alquiler residencial'
        WHEN p.tipo_operacion='venta' THEN 'Venta propiedad'
        ELSE ''
    END AS descripcion
FROM bootcamp.silver.propiedades p
WHERE p.tipo_operacion IS NOT NULL AND p.moneda IS NOT NULL
ORDER BY p.tipo_operacion, p.moneda;

In [0]:
%sql
INSERT INTO bootcamp.gold.dim_tiempo
(
    fecha_id,
    fecha,
    anio,
    mes,
    trimestre,
    dia_semana,
    es_fin_de_semana
)
SELECT
    CAST(DATE_FORMAT(fecha_publicacion, 'yyyyMMdd') AS BIGINT) as fecha_id,
    fecha_publicacion AS fecha,
    YEAR(fecha_publicacion) AS anio,
    MONTH(fecha_publicacion) AS mes,
    QUARTER(fecha_publicacion) AS trimestre,
    CASE DAYOFWEEK(fecha_publicacion)
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Lunes'
        WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miercoles'
        WHEN 5 THEN 'Jueves'
        WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sabado'
    END AS dia_semana,
    DAYOFWEEK(fecha_publicacion) IN (1,7) AS es_fin_de_semana    
FROM bootcamp.silver.propiedades
WHERE fecha_publicacion is NOT NULL
GROUP BY fecha_publicacion
ORDER BY fecha;

In [0]:
%sql
SELECT DISTINCT estado FROM bootcamp.silver.propiedades;

In [0]:
%sql
SELECT DISTINCT cochera FROM bootcamp.silver.propiedades;

In [0]:
%sql

INSERT INTO bootcamp.gold.dim_caracteristicas
(
    estado,
    cochera
)
SELECT
    DISTINCT
    COALESCE(estado,'sin especificar') as estado,
    COALESCE(cochera,false) as cochera
FROM bootcamp.silver.propiedades
ORDER BY estado, cochera;

In [0]:
SELECT DISTINCT orientacion FROM bootcamp.silver.propiedades ORDER BY orientacion;

In [0]:
%sql
INSERT INTO bootcamp.gold.dim_orientacion
(
    orientacion,
    tipo_orientacion
)
SELECT DISTINCT
    COALESCE(p.orientacion, 'sin especificar') AS orientacion,
    CASE
        WHEN p.orientacion = 'norte' THEN 'Norte'
        WHEN p.orientacion = 'sur' THEN 'Sur'
        WHEN p.orientacion = 'este' THEN 'Este'
        WHEN p.orientacion ='oeste' THEN 'Oeste'
        ELSE 'Sin especificar'
    END AS tipo_orientacion
FROM bootcamp.silver.propiedades p
ORDER BY orientacion;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW temp_gold AS 
    SELECT /*+ BROADCAST(z, to, f, c, o) */
     MD5(CONCAT_WS('|', p.url, CAST(p.precio AS STRING))) AS row_hash,
     z.zona_id,
     to.tipo_operacion_id,
     f.fecha_id,
     c.caracteristicas_id,
     o.orientacion_id,          
     p.url,
     p.precio,
     p.expensas,
     p.precio_por_m2,
     p.metros_cuadrados_totales,
     p.metros_cuadrados_cubiertos,
     p.ambientes
FROM bootcamp.silver.propiedades p
LEFT JOIN bootcamp.gold.dim_zona z ON p.partido=z.partido AND p.region=z.region
LEFT JOIN bootcamp.gold.dim_tipo_operacion to ON p.tipo_operacion=to.tipo_operacion AND p.moneda=to.moneda
LEFT JOIN bootcamp.gold.dim_tiempo f ON p.fecha_publicacion=f.fecha
LEFT JOIN bootcamp.gold.dim_caracteristicas c ON p.estado=c.estado AND p.cochera=c.cochera
LEFT JOIN bootcamp.gold.dim_orientacion o ON p.orientacion=o.orientacion;

SELECT COUNT(*) AS total_reg_tmp_view FROM temp_gold;

In [0]:
%sql
-- 1. Revisar duplicados en dim_zona
SELECT partido, region, COUNT(*) 
FROM bootcamp.gold.dim_zona 
GROUP BY partido, region 
HAVING COUNT(*) > 1;
-- sin duplicados

-- 2. Revisar duplicados en dim_tipo_operacion
SELECT tipo_operacion, moneda, COUNT(*) 
FROM bootcamp.gold.dim_tipo_operacion 
GROUP BY tipo_operacion, moneda 
HAVING COUNT(*) > 1;
--sin duplicados

-- 3. Revisar duplicados en dim_caracteristicas
SELECT estado, cochera, COUNT(*) 
FROM bootcamp.gold.dim_caracteristicas 
GROUP BY estado, cochera 
HAVING COUNT(*) > 1;
-- sin duplicados

-- 4. Revisar duplicados en dim_orientacion
SELECT orientacion, COUNT(*) 
FROM bootcamp.gold.dim_orientacion 
GROUP BY orientacion 
HAVING COUNT(*) > 1;
--sin duplicados

-- 5. Revisar duplicados en dim_tiempo
SELECT fecha, COUNT(*) 
FROM bootcamp.gold.dim_tiempo 
GROUP BY fecha 
HAVING COUNT(*) > 1;


In [0]:
%sql
-- WITH data AS (
--     SELECT /*+ BROADCAST(z, to, f, c, o) */
--      MD5(CONCAT_WS('|', p.url, CAST(p.precio AS STRING))) AS row_hash,
--      z.zona_id,
--      to.tipo_operacion_id,
--      f.fecha_id,
--      c.caracteristicas_id,
--      o.orientacion_id,          
--      p.url,
--      p.precio,
--      p.expensas,
--      p.precio_por_m2,
--      p.metros_cuadrados_totales,
--      p.metros_cuadrados_cubiertos,
--      p.ambientes
-- FROM bootcamp.silver.propiedades p
-- LEFT JOIN bootcamp.gold.dim_zona z ON p.partido=z.partido AND p.region=z.region
-- LEFT JOIN bootcamp.gold.dim_tipo_operacion to ON p.tipo_operacion=to.tipo_operacion AND p.moneda=to.moneda
-- LEFT JOIN bootcamp.gold.dim_tiempo f ON p.fecha_publicacion=f.fecha
-- LEFT JOIN bootcamp.gold.dim_caracteristicas c ON p.estado=c.estado AND p.cochera=c.cochera
-- LEFT JOIN bootcamp.gold.dim_orientacion o ON p.orientacion=o.orientacion
-- )
INSERT INTO bootcamp.gold.fact_propiedades
(
    row_hash,
    zona_id,
    tipo_orientacion_id,
    fecha_id,
    caracteristicas_id,
    tipo_operacion_id,
    url,
    precio,
    expensas,
    precio_por_m2,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    ambientes
)
SELECT * FROM temp_gold;
-- SELECT /*+ BROADCAST(z, to, f, c, o) */
--      MD5(CONCAT_WS('|', p.url, CAST(p.precio AS STRING))) AS row_hash,
--      z.zona_id,
--      to.tipo_operacion_id,
--      f.fecha_id,
--      c.caracteristicas_id,
--      o.orientacion_id,          
--      p.url,
--      p.precio,
--      p.expensas,
--      p.precio_por_m2,
--      p.metros_cuadrados_totales,
--      p.metros_cuadrados_cubiertos,
--      p.ambientes
-- FROM bootcamp.silver.propiedades p
-- LEFT JOIN bootcamp.gold.dim_zona z ON p.partido=z.partido AND p.region=z.region
-- LEFT JOIN bootcamp.gold.dim_tipo_operacion to ON p.tipo_operacion=to.tipo_operacion AND p.moneda=to.moneda
-- LEFT JOIN bootcamp.gold.dim_tiempo f ON p.fecha_publicacion=f.fecha
-- LEFT JOIN bootcamp.gold.dim_caracteristicas c ON p.estado=c.estado AND p.cochera=c.cochera
-- LEFT JOIN bootcamp.gold.dim_orientacion o ON p.orientacion=o.orientacion;


In [0]:
%sql

SELECT
    row_hash,
    COUNT(*) AS cantidad
FROM bootcamp.gold.fact_propiedades
GROUP BY row_hash
HAVING cantidad > 1;
